[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/megano/deep-learning-image-classifier/blob/master/snowman_train.ipynb)

# Snowman Image Classifier — Training

This notebook builds an image classifier that can tell the difference between a friendly snowman (Olaf from Disney's *Frozen*) and a hostile snowman. Downloads training images from curated URL lists, fine-tunes ResNet-18 using fast.ai, and exports the model for the demo notebook. See the repo README for project background.

## 1. Install and import dependencies

In [ ]:
import warnings
warnings.filterwarnings('ignore')

!pip install -q fastai

from fastai.vision.all import *
from fastai.vision.widgets import ImageClassifierCleaner
import shutil

In [ ]:
# Fetch curated URL lists from the repo (not bundled with the notebook on Colab).
!mkdir -p data
!wget -q https://raw.githubusercontent.com/megano/deep-learning-image-classifier/master/data/urls_olaf.csv -O data/urls_olaf.csv
!wget -q https://raw.githubusercontent.com/megano/deep-learning-image-classifier/master/data/urls_hostile.csv -O data/urls_hostile.csv

## 2. Download training images

Both classes use curated URL lists stored in the repo. This avoids rate limits from image search APIs on Colab.

Why so few images? Transfer learning. Rather than training from scratch (which requires thousands of examples), we start with ResNet-18, a model pre-trained on the ILSVRC subset of ImageNet (~1.2 million images). It already understands edges, shapes, and textures. We only need enough new images to teach it the specific distinction we care about.

In [ ]:
def load_urls_from_file(filepath, max_urls=150):
    """Load image URLs from a CSV file (one URL per line)."""
    with open(filepath) as f:
        urls = [line.strip() for line in f if line.strip()]
    return L(urls[:max_urls])

In [ ]:
# Run this cell to clear downloaded images and start fresh.
# Needed after a runtime restart since the download cell skips if the folder exists.
shutil.rmtree('snowman', ignore_errors=True)

In [ ]:
path = Path('snowman')

if not path.exists():
    path.mkdir()

    for folder, csv_file in [('olaf', 'data/urls_olaf.csv'), ('hostile', 'data/urls_hostile.csv')]:
        dest = path/folder
        dest.mkdir(exist_ok=True)
        urls = load_urls_from_file(csv_file, max_urls=150)
        download_images(dest, urls=urls)
        print(f'Downloaded {len(get_image_files(dest))} images for: {folder}')

In [ ]:
# Some downloaded images may be corrupt or in unsupported formats.
# verify_images checks each file and returns a list of bad ones.
fns = get_image_files(path)
failed = verify_images(fns)
failed.map(Path.unlink)
print(f'Removed {len(failed)} corrupt images. {len(get_image_files(path))} remaining.')

## 3. Build the DataBlock and inspect the data

The first step in any data problem is to look at the data. We need to understand what we have before we can train on it.

fast.ai's DataBlock API lets us describe how to load and label our data in a flexible, reusable way. We tell it:
- What kind of inputs and outputs we have (images and categories)
- How to split into train/validation sets
- How to get labels (from the parent folder name)
- What transforms to apply (random crops and augmentations to improve generalization)

In [ ]:
snowman = DataBlock(
    blocks=(ImageBlock, CategoryBlock),       # inputs are images, outputs are categories
    get_items=get_image_files,                # how to find the files
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # 80/20 train/validation split
    get_y=parent_label,                       # label = parent folder name (olaf / hostile)
    item_tfms=RandomResizedCrop(224, min_scale=0.5),  # randomly crop 50-100% of the image, resize to 224x224
    batch_tfms=aug_transforms()               # random flips, rotations, lighting changes
)

dls = snowman.dataloaders(path)

In [ ]:
# Sanity check: view a sample batch to confirm labels and image quality.
dls.valid.show_batch(max_n=6, nrows=2)

## 4. Train the model

We use `vision_learner` to create a ResNet-18 model with a new classification head for our two classes.

`fine_tune(4)` runs fast.ai's two-phase training:
1. **Freeze** (1 epoch): pre-trained layers are frozen. Only the new head (the classification layer) is trained. This adapts the output to our specific classes without destroying the pre-trained weights.
2. **Unfreeze** (4 epochs): all layers are unfrozen and trained together, using discriminative learning rates: smaller rates for early layers (which already understand basic features) and larger rates for later layers.

That's 5 total epochs of training.

In [ ]:
# vision_learner creates a CNN using a pre-trained backbone (resnet18).
# error_rate tracks the fraction of validation images classified incorrectly.
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

## 5. Interpret results

The confusion matrix shows where the model makes mistakes. For a binary classifier, we want to see high numbers on the diagonal (correct predictions) and low numbers off-diagonal (errors).

`plot_top_losses` shows the images where the model performed worst, which is useful for spotting mislabeled or ambiguous images in the training set.

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
# Top losses = images with the highest loss, which includes both incorrect
# predictions, and correct predictions where the model had low confidence.
interp.plot_top_losses(5, nrows=1)

## 5b. Clean the data (optional)

Use the `ImageClassifierCleaner` widget to review images sorted by loss. Mark junk images for deletion or reclassify mislabeled ones. Review each class/split combo, then run the apply cell. After cleanup, rerun from section 3 to retrain on cleaner data.

In [ ]:
cleaner = ImageClassifierCleaner(learn)
cleaner

In [ ]:
# Apply cleanup (run after reviewing each class/split combo)
for idx in cleaner.delete(): cleaner.fns[idx].unlink(missing_ok=True)
for idx,cat in cleaner.change(): shutil.move(str(cleaner.fns[idx]), path/cat)
print(f'{len(get_image_files(path))} images remaining.')

## 6. Export the model

`learn.export()` saves the model weights and all preprocessing steps (transforms, label mappings) into a single `export.pkl` file. This is everything needed to run inference, no training code required.

The exported model is used by `snowman_demo.ipynb` for the interactive classifier.

In [ ]:
learn.export()

# Confirm the file was saved
path = Path()
path.ls(file_exts='.pkl')